In [9]:
from google import genai
from PIL import Image

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [10]:
# 1. Initialize the client (automatically picks up GEMINI_API_KEY from environment)
client = genai.Client(
    vertexai=True,
    project="infinitas-ds-ai-poc",
    location="global",  # changed from us-central1 previousy bc no access to gemini 3.5
)

### Put down the image file paths in batches

- Seperate into batches for ease of analysis (each batch kinda has their own charateristics so it's easy to improve my prompts if some batches perform poorly)

In [11]:
# 2. Open your local image file

path_batch1 = ["test_images/First_Tamper.png", "test_images/first-tamper.png", "test_images/pink.png","test_images/combined.png",
         "test_images/neat.png", "test_images/neatest.png", "test_images/cropped-obvious.png"] # explore strength and weaknesses

path_batch2 = ["test_images/payslip1.jpeg", "test_images/payslip10.jpeg", "test_images/payslip2.png",
               "test_images/payslip20.png","test_images/payslip3.png", "test_images/payslip30.png"] # explore payslip with white space but different fonts

paths = path_batch2
images = []

for path in paths:
    images.append(Image.open(path))

print(len(images))

6


## Ground truth labels

Set the true `label` ("tampered" / "authentic") and `label_signals` (true signal types present, if tampered) for each image in this batch. Stored in `image_labels.json`, keyed by image path.

`log_result()` looks these up by `image_path` automatically when logging — they are for analysis only and are **never** included in the prompt/model call. Run this once per image (re-running just overwrites with the same value).

In [12]:
# # Only need to set up after making a new image once after it's been created.

# from result_logger import set_image_label, load_image_labels

# # Fill in the real ground truth for each image in `paths` below.
# # label: "tampered" or "authentic"
# # label_signals: list of signal types actually present (only meaningful if "tampered")
# #   -- see EXPECTED_SIGNAL_TYPES in result_logger.py for valid values

# ground_truth: dict[str, tuple[str, list[str]]] = {
#     paths[0]: ("authentic", []),
#     paths[1]: ("tampered", ["background_overlay", "font_weight_inconsistency",]),
#     paths[2]: ("tampered", ["background_overlay", "font_weight_inconsistency"]),
#     paths[3]: ("tampered", ["background_overlay", "font_weight_inconsistency"]),
#     paths[4]: ("tampered", ["font_weight_inconsistency","baseline_mismatch"]),
#     paths[5]: ("tampered", ["font_weight_inconsistency"]),
#     paths[6]: ("tampered", ["background_overlay", "font_weight_inconsistency", "baseline_mismatch"]),
# }

# for image_path, (label, label_signals) in ground_truth.items():
#     set_image_label(image_path, label, label_signals)

# load_image_labels()

In [13]:
# Read the V1 prompt

with open("prompt-library/V1.txt", "r", encoding="utf-8") as file:
    file_content = file.read()

prompt_v1 = file_content
print(type(prompt_v1))

# Read the V1 prompt split into system instruction + task prompt

with open("prompt-library/V1_system.txt", "r", encoding="utf-8") as file:
    system_prompt_v1 = file.read()

with open("prompt-library/V1_task.txt", "r", encoding="utf-8") as file:
    task_prompt_v1 = file.read()

# Read the V2 prompt (combined) and its system/task split, same pattern as V1

with open("prompt-library/V2.txt", "r", encoding="utf-8") as file:
    prompt_v2 = file.read()

with open("prompt-library/V2_system.txt", "r", encoding="utf-8") as file:
    system_prompt_v2 = file.read()

with open("prompt-library/V2_task.txt", "r", encoding="utf-8") as file:
    task_prompt_v2 = file.read()

<class 'str'>


## Result logging

Every model call below is logged to `notebook_results/results_log.jsonl` via `result_logger.log_result()`.
This keeps a permanent record across batches without cluttering this notebook.

**For each new batch of images:**
1. Bump `BATCH_ID` below (e.g. `"batch2"`)
2. Update `paths` in the image-loading cell to point at the new images
3. Re-run the experiment cells as usual — results from previous batches stay in the log

Use the "Review logged results" section at the end to compare across batches.

## Try with the same model for now to test the workflow
### Models (to double check)
- "gemini-2.5-flash"
- "gemini-2.5-pro"
- "gemini-3.5-flash"
- "gemini-3.1-pro-preview" because "gemini-3.5-pro" was not yet available on 12/06/2026 but releasing very soon

In [14]:
from exp_running import PromptVersion, run_experiment

# All prompt variants to sweep over. Add new versions here as new entries.
prompt_versions = [
    PromptVersion(prompt_id="V1", mode="combined", content=prompt_v1),
    PromptVersion(prompt_id="V1_split", mode="split", system=system_prompt_v1, task=task_prompt_v1),
    PromptVersion(prompt_id="V2", mode="combined", content=prompt_v2),
    PromptVersion(prompt_id="V2_split", mode="split", system=system_prompt_v2, task=task_prompt_v2),
]

# Sweep config: edit these lists to control which combinations get run.
models = ["gemini-3.5-flash", "gemini-3.1-pro-preview"]  # ["gemini-2.5-flash", "gemini-2.5-pro", "gemini-3.5-flash", "gemini-3.1-pro-preview"]
temperatures = [0.1] #[0, 0.1, 0.2,0.8]

image_list: list[tuple[str, Image.Image]] = list(zip(paths, images))

In [16]:
BATCH_ID = "batch2"  # bump this for each new set of test images

# Run the full sweep: every image x every prompt version x every model x every temperature.
for model in models:
    for image_path, image in image_list:
        for prompt_version in prompt_versions:
            for temperature in temperatures:
                run_experiment(client, image_path, image, prompt_version, model, temperature, BATCH_ID)

test_images/payslip1.jpeg | V1 | gemini-3.5-flash | T=0.1 -> logged (12.07s)
test_images/payslip1.jpeg | V1_split | gemini-3.5-flash | T=0.1 -> logged (8.29s)
test_images/payslip1.jpeg | V2 | gemini-3.5-flash | T=0.1 -> logged (21.04s)
test_images/payslip1.jpeg | V2_split | gemini-3.5-flash | T=0.1 -> logged (16.82s)
test_images/payslip10.jpeg | V1 | gemini-3.5-flash | T=0.1 -> logged (8.02s)
test_images/payslip10.jpeg | V1_split | gemini-3.5-flash | T=0.1 -> logged (6.85s)
test_images/payslip10.jpeg | V2 | gemini-3.5-flash | T=0.1 -> logged (13.94s)
test_images/payslip10.jpeg | V2_split | gemini-3.5-flash | T=0.1 -> logged (9.4s)
test_images/payslip2.png | V1 | gemini-3.5-flash | T=0.1 -> logged (19.9s)
test_images/payslip2.png | V1_split | gemini-3.5-flash | T=0.1 -> logged (8.89s)
test_images/payslip2.png | V2 | gemini-3.5-flash | T=0.1 -> logged (17.29s)
test_images/payslip2.png | V2_split | gemini-3.5-flash | T=0.1 -> logged (14.03s)
test_images/payslip20.png | V1 | gemini-3.5-fla

## Review logged results

Load the full log (all batches) or just the current `BATCH_ID` to compare prompts/models without re-running anything.

In [17]:
import pandas as pd
from result_logger import load_results

df_all = load_results() # batch_id=BATCH_ID
df_all['timestamp'] = pd.to_datetime(df_all['timestamp'])
#df_all[["timestamp", "batch_id", "image_path", "prompt_id", "model", "verdict", "confidence", "signal_types", "format", "latency_s"]]

In [18]:
df_all['model'].value_counts()

model
gemini-2.5-flash          66
gemini-2.5-pro            56
gemini-3.5-flash          52
gemini-3.1-pro-preview    52
Name: count, dtype: int64

In [15]:
df_all.columns

Index(['timestamp', 'batch_id', 'image_path', 'prompt_id', 'model',
       'temperature', 'latency_s', 'raw_response', 'parsed_response', 'notes',
       'label', 'label_signals', 'verdict', 'confidence', 'signal_types',
       'format'],
      dtype='object')

In [16]:
df_all.info()

<class 'pandas.core.frame.DataFrame'>
Index: 72 entries, 0 to 121
Data columns (total 16 columns):
 #   Column           Non-Null Count  Dtype              
---  ------           --------------  -----              
 0   timestamp        72 non-null     datetime64[ns, UTC]
 1   batch_id         72 non-null     object             
 2   image_path       72 non-null     object             
 3   prompt_id        72 non-null     object             
 4   model            72 non-null     object             
 5   temperature      72 non-null     float64            
 6   latency_s        72 non-null     float64            
 7   raw_response     72 non-null     object             
 8   parsed_response  72 non-null     object             
 9   notes            72 non-null     object             
 10  label            72 non-null     object             
 11  label_signals    72 non-null     object             
 12  verdict          72 non-null     object             
 13  confidence       72 non-nu

In [8]:
df_all[df_all['batch_id'] == 'batch1']

,timestamp,batch_id,image_path,prompt_id,model,temperature,latency_s,raw_response,parsed_response,notes,label,label_signals,verdict,confidence,signal_types,format
0,2026-06-11T02:20:36.332953+00:00,batch1,test_images/First_Tamper.png,V1,gemini-2.5-flash,0.10,9.42,"```json\n{\n ""verdict"": ""authentic"",\n ""conf...","{'verdict': 'authentic', 'confidence': 'high',...",,authentic,[],authentic,high,[],True
1,2026-06-11T02:20:47.181659+00:00,batch1,test_images/first-tamper.png,V1,gemini-2.5-flash,0.10,10.85,"```json\n{\n ""verdict"": ""tampered"",\n ""confi...","{'verdict': 'tampered', 'confidence': 'medium'...",,tampered,"[background_overlay, font_weight_inconsistency]",tampered,medium,[resolution_mismatch],True
2,2026-06-11T02:20:53.951337+00:00,batch1,test_images/First_Tamper.png,V1_split,gemini-2.5-flash,0.10,6.76,"```json\n{\n ""verdict"": ""authentic"",\n ""conf...","{'verdict': 'authentic', 'confidence': 'high',...",,authentic,[],authentic,high,[],True
3,2026-06-11T02:21:04.694144+00:00,batch1,test_images/first-tamper.png,V1_split,gemini-2.5-flash,0.10,10.74,"```json\n{\n ""verdict"": ""authentic"",\n ""conf...","{'verdict': 'authentic', 'confidence': 'high',...",,tampered,"[background_overlay, font_weight_inconsistency]",authentic,high,[],True
4,2026-06-11T02:24:43.831825+00:00,batch1,test_images/First_Tamper.png,V1,gemini-2.5-flash,0.05,9.18,"```json\n{\n ""verdict"": ""authentic"",\n ""conf...","{'verdict': 'authentic', 'confidence': 'high',...",,authentic,[],authentic,high,[],True
5,2026-06-11T02:24:51.311197+00:00,batch1,test_images/first-tamper.png,V1,gemini-2.5-flash,0.05,7.48,"```json\n{\n ""verdict"": ""authentic"",\n ""conf...","{'verdict': 'authentic', 'confidence': 'high',...",,tampered,"[background_overlay, font_weight_inconsistency]",authentic,high,[],True
6,2026-06-11T02:25:08.611948+00:00,batch1,test_images/combined.png,V1,gemini-2.5-flash,0.05,17.30,"```json\n{\n ""verdict"": ""tampered"",\n ""confi...","{'verdict': 'tampered', 'confidence': 'high', ...",,tampered,"[background_overlay, font_weight_inconsistency]",tampered,high,"[background_anomaly, background_anomaly]",False
7,2026-06-11T02:25:16.513412+00:00,batch1,test_images/neat.png,V1,gemini-2.5-flash,0.05,7.90,"```json\n{\n ""verdict"": ""authentic"",\n ""conf...","{'verdict': 'authentic', 'confidence': 'high',...",,tampered,"[font_weight_inconsistency, baseline_mismatch]",authentic,high,[],True
8,2026-06-11T02:25:28.105398+00:00,batch1,test_images/neatest.png,V1,gemini-2.5-flash,0.05,11.59,"```json\n{\n ""verdict"": ""tampered"",\n ""confi...","{'verdict': 'tampered', 'confidence': 'medium'...",,tampered,[font_weight_inconsistency],tampered,medium,[font_weight_inconsistency],True
9,2026-06-11T02:25:42.508209+00:00,batch1,test_images/cropped-obvious.png,V1,gemini-2.5-flash,0.05,14.40,"```json\n{\n ""verdict"": ""authentic"",\n ""conf...","{'verdict': 'authentic', 'confidence': 'high',...",,tampered,"[background_overlay, font_weight_inconsistency...",authentic,high,[],True


In [9]:
# Calculate some accuracy, recall, F1 metrics
total_tests = len(df_all)
total_wrong_verdict = len(df_all[df_all['label'] != df_all['verdict']])

print("Wrong_verdict_percent", total_wrong_verdict/total_tests*100)

Wrong_verdict_percent 29.78723404255319


### Different batches will be different I am sure

In [10]:
df_all['verdict'].value_counts()

verdict
tampered        52
authentic       41
inconclusive     1
Name: count, dtype: int64

In [11]:
df_all[(df_all['verdict'] == 'inconclusive')]

,timestamp,batch_id,image_path,prompt_id,model,temperature,latency_s,raw_response,parsed_response,notes,label,label_signals,verdict,confidence,signal_types,format
42,2026-06-11T06:06:48.188613+00:00,batch2,test_images/payslip1.jpeg,V1,gemini-2.5-flash,0.05,31.21,"```json\n{\n ""verdict"": ""inconclusive"",\n ""c...","{'verdict': 'inconclusive', 'confidence': 'hig...",,tampered,"[font_weight_inconsistency, font_size_mismatch...",inconclusive,high,[font_weight_inconsistency],True


In [20]:
from analysis_util import compute_binary_metrics
total_matrics = compute_binary_metrics(df_all)
print(total_matrics)

precision           0.905512
recall              0.714286
specificity         0.812500
f1                  0.798611
accuracy            0.742222
true_positive     115.000000
false_positive     12.000000
false_negative     46.000000
true_negative      52.000000
excluded_count      1.000000
dtype: float64


#### Batch 1

In [21]:
batch1 = df_all[df_all['batch_id'] =='batch1']
print(compute_binary_metrics(batch1))

precision          0.975309
recall             0.731481
specificity        0.900000
f1                 0.835979
accuracy           0.757812
true_positive     79.000000
false_positive     2.000000
false_negative    29.000000
true_negative     18.000000
excluded_count     0.000000
dtype: float64


#### Batch 2

In [22]:
batch2 = df_all[df_all['batch_id'] =='batch2']
print(compute_binary_metrics(batch2))

precision          0.782609
recall             0.679245
specificity        0.772727
f1                 0.727273
accuracy           0.721649
true_positive     36.000000
false_positive    10.000000
false_negative    17.000000
true_negative     34.000000
excluded_count     1.000000
dtype: float64


#### Compare different prompt versions

In [23]:
df_all['prompt_id'].value_counts()

prompt_id
V1          86
V1_split    74
V2          33
V2_split    33
Name: count, dtype: int64

In [24]:
df_all.groupby('prompt_id').apply(compute_binary_metrics)

/var/folders/1k/s8c8mkvd19307rbdq3mhs23h0000gp/T/ipykernel_9037/994550254.py:1: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_all.groupby('prompt_id').apply(compute_binary_metrics)


,precision,recall,specificity,f1,accuracy,true_positive,false_positive,false_negative,true_negative,excluded_count
prompt_id,,,,,,,,,,
V1,0.893617,0.711864,0.807692,0.792453,0.741176,42.0,5.0,17.0,21.0,1.0
V1_split,0.880952,0.685185,0.750000,0.770833,0.702703,37.0,5.0,17.0,15.0,0.0
V2,0.944444,0.708333,0.888889,0.809524,0.757576,17.0,1.0,7.0,8.0,0.0
V2_split,0.950000,0.791667,0.888889,0.863636,0.818182,19.0,1.0,5.0,8.0,0.0


In [25]:
# Latency check: Latency is better?, To check again when have more results
print(df_all.groupby(['prompt_id'])['latency_s'].mean())
print(df_all.groupby(['prompt_id'])['latency_s'].std())

prompt_id
V1          15.969419
V1_split    14.562703
V2          19.570303
V2_split    17.193636
Name: latency_s, dtype: float64
prompt_id
V1           9.517979
V1_split     6.397656
V2          21.830507
V2_split     7.982926
Name: latency_s, dtype: float64


#### Stability analysis
- Let's see how often do each model/ per batch/ per temperature change their predictions

In [26]:
df_all.columns

Index(['timestamp', 'batch_id', 'image_path', 'prompt_id', 'model',
       'temperature', 'latency_s', 'raw_response', 'parsed_response', 'notes',
       'label', 'label_signals', 'verdict', 'confidence', 'signal_types',
       'format'],
      dtype='object')

In [28]:
df_all['image_path'].value_counts()

image_path
test_images/First_Tamper.png       22
test_images/first-tamper.png       22
test_images/combined.png           20
test_images/neat.png               20
test_images/neatest.png            20
test_images/cropped-obvious.png    20
test_images/pink.png               18
test_images/payslip1.jpeg          14
test_images/payslip10.jpeg         14
test_images/payslip2.png           14
test_images/payslip20.png          14
test_images/payslip3.png           14
test_images/payslip30.png          14
Name: count, dtype: int64

In [30]:
verdict_proportions = df_all.groupby('image_path')['verdict'].value_counts(normalize=True)
verdict_proportions

image_path                       verdict     
test_images/First_Tamper.png     authentic       0.909091
                                 tampered        0.090909
test_images/combined.png         tampered        1.000000
test_images/cropped-obvious.png  tampered        0.900000
                                 authentic       0.100000
test_images/first-tamper.png     tampered        0.818182
                                 authentic       0.181818
test_images/neat.png             authentic       0.550000
                                 tampered        0.450000
test_images/neatest.png          authentic       0.600000
                                 tampered        0.400000
test_images/payslip1.jpeg        authentic       0.642857
                                 tampered        0.285714
                                 inconclusive    0.071429
test_images/payslip10.jpeg       authentic       0.857143
                                 tampered        0.142857
test_images/payslip2.png  